# Import Dependencies

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import joblib
import pandas as pd

from src.features import add_band_availability, add_cation_ratio, add_missing_indicators
from src.modeling.inference import prepare_inference_data
from src.modeling.train import build_catboost

# Load Data & Reproduksi Feature Set

In [3]:
# Load dataset
df = pd.read_csv("../data/raw/train.csv", index_col="sample_id")
df.head()

,source_id,has_band_A_spectrum,has_band_B_spectrum,sampling_strategy,sampling_depth_cm,geo_zone_macro,geo_zone_micro,geo_zone_meso,land_cover_type,biome,...,spectral_band_B_PC_6,spectral_band_B_PC_7,spectral_band_B_PC_8,spectral_band_B_PC_9,spectral_band_B_PC_10,spectral_band_B_PC_11,spectral_band_B_PC_12,spectral_band_B_PC_13,spectral_band_B_PC_14,spectral_band_B_PC_15
sample_id,,,,,,,,,,,,,,,,,,,,,
train_00001,Source_01,YES,NO,Auger,0-20,SE,Unknown,State_01,Seasonal Semideciduous Forest,Mata Atlantica,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
train_00002,Source_10,YES,NO,Auger,0-20,MW,Loc_011,State_10,Savannah,Cerrado,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
train_00003,Source_04,YES,NO,Auger,0-20,S,Loc_001,State_06,Unknown,Unknown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
train_00004,Source_02,YES,NO,Auger,0-20,N,Unknown,State_07,Unknown,Amazonia,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
train_00005,Source_04,YES,NO,Auger,0-20,S,Loc_001,State_06,Unknown,Unknown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Pembuatan fitur-fitur tambahan
df = add_missing_indicators(df)
df = add_band_availability(df)
df = add_cation_ratio(df)

In [5]:
X = df.drop(columns="property_organic_content")
y = df["property_organic_content"]

# Categorical fetures
cat_feats = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

# Load Study & Rekonstruksi Best Params

In [6]:
study = joblib.load("../outputs/study.pkl")

final_params = {"eval_metric": "RMSE", **study.best_params}

c:\Users\ASUS\miniconda3\envs\compfest_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Fit Final Model

In [7]:
final_model = build_catboost(final_params)
final_model.fit(
    X,
    y,
    cat_features=cat_feats,
)

CatBoostRegressor(bootstrap_type='Bayesian', depth=9, eval_metric='RMSE', iterations=1000, learning_rate=0.08966956777379233, loss_function='RMSE', min_data_in_leaf=81, random_state=42, verbose=False)

# Simpan Model Artifact

In [8]:
Path("../outputs").mkdir(exist_ok=True)

joblib.dump(
    {
        "model": final_model,
        "cat_feats": cat_feats,
        "params": final_params,
    },
    "../outputs/final_model_bundle.pkl",
)

['../outputs/final_model_bundle.pkl']

# Generate Submission

## Load & Preprocessing Test Set  

In [9]:
df_test = pd.read_csv("../data/raw/test.csv", index_col="sample_id")

# Simpan sample_id terpisah sebelum diproses, untuk submission nanti
test_sample_ids = df_test.index.copy()

X_test = prepare_inference_data(df_test)

print(f"X_test shape: {X_test.shape}")
assert list(X_test.columns) == [c for c in X.columns], "Kolom X_test tidak sama dengan X_full —-> cek pipeline"

X_test shape: (2670, 59)


## Predict & Simpan submission.csv

In [10]:
preds_test = final_model.predict(X_test)

submission = pd.DataFrame({
    "sample_id": test_sample_ids,
    "property_organic_content": preds_test,
})

submission.to_csv("../outputs/submission.csv", index=False)
submission.head()

,sample_id,property_organic_content
0,test_00001,23.801200
1,test_00002,9.349625
2,test_00003,30.531373
3,test_00004,25.292308
4,test_00005,28.778546
